In [1]:
from google.colab import files

uploaded = files.upload()

for fn in uploaded.keys():
  print(f'User uploaded file "{fn}" with length {len(uploaded[fn])} bytes')

Saving recruitment_data.csv to recruitment_data.csv
User uploaded file "recruitment_data.csv" with length 63964 bytes


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, roc_curve, auc
)
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans

# Set display options for cleaner output
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

# --------------------------------------------------------------------------------------------------
# PART 1: DATA LOADING AND PREPROCESSING
# --------------------------------------------------------------------------------------------------

print("--- Starting ML Pipeline ---")

# 1. Load Data
df = pd.read_csv('recruitment_data.csv')
print(f"Initial Data Loading Complete. Dataset Shape: {df.shape}")

# Separate Features (X) and Target (y)
X = df.drop('HiringDecision', axis=1)
y = df['HiringDecision']

# 2. Handle Categorical Features (One-Hot Encoding)
nominal_features = ['Gender', 'RecruitmentStrategy']
X_encoded = pd.get_dummies(X, columns=nominal_features, drop_first=True)
# EducationLevel is treated as ordinal and kept as is.

# Define Numerical Features for Scaling
numerical_features = [
    'Age', 'EducationLevel', 'ExperienceYears', 'PreviousCompanies',
    'DistanceFromCompany', 'InterviewScore', 'SkillScore', 'PersonalityScore'
]

# 3. Split Data (Crucial to split before scaling to prevent data leakage)
# stratify=y ensures the target variable's class distribution is maintained across splits
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y, test_size=0.25, random_state=42, stratify=y
)

# 4. Apply Feature Scaling (StandardScaler)
scaler = StandardScaler()
# Fit the scaler ONLY on the training data
X_train[numerical_features] = scaler.fit_transform(X_train[numerical_features])
# Transform the test data using the fitted training scaler
X_test[numerical_features] = scaler.transform(X_test[numerical_features])

print(f"Preprocessing Complete. Training Set Shape: {X_train.shape}")
print(f"Test Set Shape: {X_test.shape}")
print("-" * 30)


# --------------------------------------------------------------------------------------------------
# PART 2: EXPLORATORY DATA ANALYSIS (EDA) VISUALIZATIONS
# --------------------------------------------------------------------------------------------------

# 5. Target Distribution Plot
plt.figure(figsize=(6, 4))
sns.countplot(x='HiringDecision', data=df)
plt.title('Target Variable Distribution (HiringDecision)')
plt.xlabel('HiringDecision (0: Not Hired, 1: Hired)')
plt.ylabel('Count')
plt.savefig('target_distribution.png')
plt.close()

# 6. Correlation Heatmap and Printout
correlation_matrix = df.corr()
plt.figure(figsize=(10, 8))
sns.heatmap(correlation_matrix, annot=False, cmap='coolwarm', fmt=".2f", linewidths=.5)
plt.title('Feature Correlation Heatmap')
plt.savefig('correlation_heatmap.png')
plt.close()

target_correlations = correlation_matrix['HiringDecision'].sort_values(ascending=False)
print("\n--- Top Linear Correlations with HiringDecision (for Report) ---")
print(target_correlations)
print("-" * 30)

# 7. Box Plots for Key Scores vs. Hiring Decision
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Scores Distribution by Hiring Decision')
sns.boxplot(ax=axes[0], x='HiringDecision', y='InterviewScore', data=df)
axes[0].set_title('Interview Score')
sns.boxplot(ax=axes[1], x='HiringDecision', y='SkillScore', data=df)
axes[1].set_title('Skill Score')
sns.boxplot(ax=axes[2], x='HiringDecision', y='PersonalityScore', data=df)
axes[2].set_title('Personality Score')
plt.tight_layout()
plt.savefig('score_boxplots.png')
plt.close()


# --------------------------------------------------------------------------------------------------
# PART 3: MODEL TRAINING AND EVALUATION (Gradient Boosting Classifier)
# --------------------------------------------------------------------------------------------------

# 8. Model Initialization and Training
gbc_model = GradientBoostingClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    random_state=42 # Ensures reproducibility
)
gbc_model.fit(X_train, y_train)

# 9. Model Prediction
y_pred = gbc_model.predict(X_test)
y_pred_proba = gbc_model.predict_proba(X_test)[:, 1]

# 10. Performance Metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print("\n--- GBC Model Performance Metrics on Test Set (for Report) ---")
print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1-Score: {f1:.4f}")

# Confusion Matrix
conf_mat = confusion_matrix(y_test, y_pred)
print("\n--- Confusion Matrix (TP, TN, FP, FN) ---")
print(conf_mat)

# 11. ROC Curve and AUC Score
fpr, tpr, thresholds = roc_curve(y_test, y_pred_proba)
roc_auc = auc(fpr, tpr)
plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (area = {roc_auc:.4f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC) Curve')
plt.legend(loc="lower right")
plt.savefig('roc_curve.png')
plt.close()
print(f"ROC Curve saved. AUC Score: {roc_auc:.4f}")

# 12. Feature Importance
feature_importance = pd.Series(gbc_model.feature_importances_, index=X_train.columns).sort_values(ascending=False)
print("\n--- Feature Importance from GBC Model (for Report) ---")
print(feature_importance)

# Visualization of Feature Importance
plt.figure(figsize=(10, 6))
sns.barplot(x=feature_importance.values, y=feature_importance.index)
plt.title('Feature Importance for Hiring Decision Prediction (GBC)')
plt.xlabel('Importance Score')
plt.ylabel('Features')
plt.tight_layout()
plt.savefig('feature_importance.png')
plt.close()
print("Feature Importance Plot saved.")
print("-" * 30)

# --------------------------------------------------------------------------------------------------
# PART 4: DIMENSIONALITY REDUCTION (PCA) AND CLUSTERING (K-Means)
# --------------------------------------------------------------------------------------------------

# 13. Apply PCA to the standardized Numerical Features
# Focus on numerical features for dimensionality analysis
numerical_data_train = X_train[numerical_features]
pca = PCA(n_components=0.95, random_state=42) # Retain 95% of variance
pca_components = pca.fit_transform(numerical_data_train)

pca_df = pd.DataFrame(data=pca_components,
                      index=numerical_data_train.index,
                      columns=[f'PC{i+1}' for i in range(pca_components.shape[1])])

print("\n--- PCA Analysis (for Report) ---")
print(f"Number of components required for 95% variance: {pca.n_components_}")
print(f"Explained Variance Ratio: {pca.explained_variance_ratio_}")

# 14. Apply K-Means Clustering on the PCA Components
# Using k=3 for three logical candidate segments (Low, Medium, High quality)
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
X_train['Candidate_Cluster'] = kmeans.fit_predict(pca_df)

print("\n--- K-Means Clustering Analysis (for Report) ---")
print("Cluster Distribution in Training Data:")
print(X_train['Candidate_Cluster'].value_counts())

# Analyze the hiring rate per cluster
cluster_analysis_df = X_train[['Candidate_Cluster']].join(y_train)
cluster_hiring_rate = cluster_analysis_df.groupby('Candidate_Cluster')['HiringDecision'].agg(['count', 'mean']).rename(columns={'mean': 'HiringRate'})
print("\nHiring Rate by Cluster (0, 1, 2):")
print(cluster_hiring_rate)

print("\n--- Script Execution Finished ---")
print("All model results printed and all necessary visualizations saved as PNG files.")

--- Starting ML Pipeline ---
Initial Data Loading Complete. Dataset Shape: (1500, 11)
Preprocessing Complete. Training Set Shape: (1125, 11)
Test Set Shape: (375, 11)
------------------------------

--- Top Linear Correlations with HiringDecision (for Report) ---
HiringDecision         1.000000
EducationLevel         0.236710
SkillScore             0.203668
PersonalityScore       0.169177
InterviewScore         0.146064
ExperienceYears        0.122494
PreviousCompanies      0.044025
Age                    0.001850
Gender                -0.002249
DistanceFromCompany   -0.016791
RecruitmentStrategy   -0.477552
Name: HiringDecision, dtype: float64
------------------------------

--- GBC Model Performance Metrics on Test Set (for Report) ---
Accuracy: 0.9120
Precision: 0.9109
Recall: 0.7931
F1-Score: 0.8479

--- Confusion Matrix (TP, TN, FP, FN) ---
[[250   9]
 [ 24  92]]
ROC Curve saved. AUC Score: 0.9115

--- Feature Importance from GBC Model (for Report) ---
RecruitmentStrategy_2    0.2